# 🤖 Digital Twin Lab

This notebook builds a **RAG-powered Digital Twin chatbot** for David Inyang-Etoh.

| Phase | What happens |
|-------|--------------|
| **1 — Build Knowledge Base** | Crawl `dinyangetoh.com` + article URLs + extract LinkedIn PDF → save as `.md` files |
| **2 — Vector Ingestion** | Ingest all `.md` files into ChromaDB via the `Ingester` module |
| **3 — RAG Chat** | Retrieve relevant chunks per query → inject as context → generate in-character responses |
| **4 — Gradio + Deploy** | Launch Gradio chat UI locally → export `app.py` for HuggingFace Spaces |

> **Reuses existing modules**: `modules/crawler.py` and `modules/ingester.py` — no new code needed for crawling or ingestion.

In [ ]:
# Cell 2: Install / audit dependencies
!uv pip install playwright beautifulsoup4 pypdf gradio openai python-dotenv chromadb langchain langchain-openai langchain-text-splitters langchain-chroma langchain-community pyyaml pydantic nest-asyncio lxml
!playwright install chromium

In [ ]:
# Cell 3: Imports
import os
import asyncio
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import nest_asyncio

from modules.crawler import Crawler, CrawlOptions, save_pages
from modules.ingester import (
    Ingester,
    IngestOptions,
    ChunkingOptions,
    VectorStoreOptions,
    EmbeddingOptions,
)

print("✅ Imports OK")

In [ ]:
# Cell 4: Environment and configuration
load_dotenv(override=True)
nest_asyncio.apply()

openai_client = OpenAI()

# ─── Personal config ──────────────────────────────────────────────────
NAME             = "David Inyang-Etoh"
PERSONAL_WEBSITE = "https://www.dinyangetoh.com"

MY_ARTICLES = [
    "https://medium.com/@dinyangetoh/how-to-build-simple-restful-api-with-nodejs-expressjs-and-mongodb-99348012925d",
    "https://www.linkedin.com/pulse/token-new-currency-david-inyang-etoh-vfjmf",
    "https://www.linkedin.com/pulse/ai-supercar-you-qualified-drive-david-inyang-etoh-u1jif",
]

LINKEDIN_PDF  = Path("me/david-inyang-etoh-linkedin.pdf")
KNOWLEDGE_DIR = Path("me/knowledge")       # output directory for all .md files
VECTOR_DB_DIR = Path("db/digital_twin_db") # separate from other labs
CHAT_MODEL    = "gpt-4o-mini"
EMBED_MODEL   = "text-embedding-3-small"

# Ensure output directories exist
(KNOWLEDGE_DIR / "website").mkdir(parents=True, exist_ok=True)
(KNOWLEDGE_DIR / "articles").mkdir(parents=True, exist_ok=True)

print(f"✅ Config ready")
print(f"   Knowledge base → {KNOWLEDGE_DIR}")
print(f"   Vector DB      → {VECTOR_DB_DIR}")
print(f"   LinkedIn PDF   → {LINKEDIN_PDF} (exists={LINKEDIN_PDF.exists()})")

---
## 🌐 Phase 1 — Build Knowledge Base

Crawl the personal website (all reachable internal pages) and each article URL individually.  
Also extract the LinkedIn PDF into markdown.  
All outputs are saved as structured `.md` files under `me/knowledge/`.

In [ ]:
# Cell 6: Crawl personal website → me/knowledge/website/
crawl_options = CrawlOptions(
    max_pages=50,             # reasonable cap for personal portfolio sites
    max_depth=3,
    min_word_count=80,        # skip navigation-only stubs
    split_spa_sections=True,  # handle SPA / single-page portfolio layouts
)

crawler = Crawler()
loop = asyncio.get_event_loop()

print(f"🌐 Crawling: {PERSONAL_WEBSITE} ...")
website_results = loop.run_until_complete(
    crawler.run(url=PERSONAL_WEBSITE, options=crawl_options)
)

summary = website_results.get("summary", {})
print(f"✅ Website crawl complete:")
for k, v in summary.items():
    print(f"   {k}: {v}")

In [ ]:
# Cell 7: Save crawled website pages as .md files
website_pages = website_results["data"]["pages"]
saved_count = save_pages(website_pages, str(KNOWLEDGE_DIR / "website"))

print(f"✅ Saved {saved_count} website pages → {KNOWLEDGE_DIR}/website/")

# Preview the filenames saved
for f in sorted((KNOWLEDGE_DIR / "website").glob("*.md")):
    print(f"   📄 {f.name}  ({f.stat().st_size:,} bytes)")

In [ ]:
# Cell 8: Crawl individual article URLs (one page each)
article_options = CrawlOptions(
    max_pages=1,   # one page per URL — we're targeting a specific article
    max_depth=0,
    min_word_count=50,
    split_spa_sections=False,
)

all_article_pages = []
failed_articles = []

for url in MY_ARTICLES:
    print(f"\n📰 Fetching: {url}")
    try:
        result = loop.run_until_complete(
            crawler.run(url=url, options=article_options)
        )
        pages = result["data"]["pages"]
        all_article_pages.extend(pages)
        print(f"   → {len(pages)} page(s) extracted, {result['summary']['total_time_s']}s")
    except Exception as e:
        print(f"   ⚠️  Failed: {e}")
        failed_articles.append(url)

print(f"\n✅ Total article pages extracted: {len(all_article_pages)}")
if failed_articles:
    print(f"⚠️  Failed URLs (likely login-gated): {len(failed_articles)}")
    for u in failed_articles:
        print(f"   - {u}")

In [ ]:
# Cell 9: Save article pages as .md files
if all_article_pages:
    saved_count = save_pages(all_article_pages, str(KNOWLEDGE_DIR / "articles"))
    print(f"✅ Saved {saved_count} article pages → {KNOWLEDGE_DIR}/articles/")
    for f in sorted((KNOWLEDGE_DIR / "articles").glob("*.md")):
        print(f"   📄 {f.name}  ({f.stat().st_size:,} bytes)")
else:
    print("⚠️  No article pages to save (all URLs may have been blocked or returned no content).")
    print("   The LinkedIn PDF knowledge will still be used for those topics.")

In [ ]:
# Cell 10: Extract LinkedIn PDF → me/knowledge/linkedin.md
def pdf_to_markdown(pdf_path: Path, person_name: str) -> str:
    """Extract text from all PDF pages and wrap in structured markdown."""
    reader = PdfReader(str(pdf_path))
    text_parts = []
    for page in reader.pages:
        text = page.extract_text()
        if text and text.strip():
            text_parts.append(text.strip())
    full_text = "\n\n".join(text_parts)
    # Wrap in structured markdown with frontmatter the Ingester can parse
    return (
        f"---\n"
        f"title: LinkedIn Profile — {person_name}\n"
        f"source: linkedin_pdf\n"
        f"breadcrumb: LinkedIn Profile\n"
        f"---\n\n"
        f"# {person_name} — LinkedIn Profile\n\n"
        f"{full_text}\n"
    )

if LINKEDIN_PDF.exists():
    linkedin_md = pdf_to_markdown(LINKEDIN_PDF, NAME)
    out_path = KNOWLEDGE_DIR / "linkedin.md"
    out_path.write_text(linkedin_md, encoding="utf-8")
    print(f"✅ LinkedIn PDF extracted → {out_path}")
    print(f"   {len(linkedin_md):,} characters, {len(linkedin_md.splitlines())} lines")
else:
    print(f"⚠️  LinkedIn PDF not found at {LINKEDIN_PDF}")
    print(f"   Expected: me/david-inyang-etoh-linkedin.pdf")

In [ ]:
# Cell 11: Inspect the full knowledge base
md_files = sorted(KNOWLEDGE_DIR.rglob("*.md"))
total_bytes = sum(f.stat().st_size for f in md_files)

print(f"📚 Knowledge base: {len(md_files)} files | {total_bytes:,} bytes total\n")
print(f"{'File':<50} {'Size':>10}")
print("-" * 62)
for f in md_files:
    rel = str(f.relative_to(KNOWLEDGE_DIR))
    print(f"{rel:<50} {f.stat().st_size:>9,} bytes")

---
## 🗄️ Phase 2 — Vector Ingestion

Ingest all `.md` files from `me/knowledge/` into a **ChromaDB vector store** using the existing `Ingester` module.  
The store is saved to `db/digital_twin_db/` (separate from other labs).

In [ ]:
# Cell 13: Configure and run the Ingester
ingest_options = IngestOptions(
    knowledge_base_path=str(KNOWLEDGE_DIR),
    chunking=ChunkingOptions(
        chunk_size=1000,
        chunk_overlap=150,
        min_chunk_chars=60,
    ),
    vector_store=VectorStoreOptions(
        persist_directory=str(VECTOR_DB_DIR),
        delete_existing_collection=True,  # fresh rebuild on each run
    ),
    embedding=EmbeddingOptions(model=EMBED_MODEL),
)

ingester = Ingester(ingest_options)

print(f"⏳ Ingesting {KNOWLEDGE_DIR} → {VECTOR_DB_DIR} ...")
vectorstore, result = ingester.ingest()

print(f"\n✅ Ingestion complete:")
print(f"   Vectors:    {result.vector_count:,}")
print(f"   Dimensions: {result.embedding_dimensions:,}")
print(f"   Persisted → {VECTOR_DB_DIR}")

In [ ]:
# Cell 14: Verify — test similarity search with sample queries
from IPython.display import Markdown, display

test_queries = [
    "What does David do professionally?",
    "What projects has David worked on?",
    "What are David's technical skills?",
]

for query in test_queries:
    hits = ingester.similarity_search(query, k=2)
    print(f"\n🔍 Query: '{query}'")
    print(f"   Top {len(hits)} chunks:")
    for i, h in enumerate(hits, 1):
        source = h.metadata.get("source", "?")
        title  = h.metadata.get("page_title", h.metadata.get("title", ""))
        preview = h.page_content[:150].replace("\n", " ")
        print(f"   [{i}] {Path(source).name} | {title}")
        print(f"       {preview}...")

---
## 💬 Phase 3 — RAG Chat

A RAG-powered chat function that:
1. Retrieves the top-k most relevant chunks from ChromaDB for each user message
2. Injects those chunks as context into the system prompt
3. Generates an in-character response as David using GPT-4o-mini

In [ ]:
# Cell 16: Base system prompt (persona — context injected dynamically per query)
BASE_SYSTEM_PROMPT = f"""You are acting as {NAME} — their digital twin.
You answer questions about {NAME}'s career, skills, projects, writing, and professional interests.
Always speak in first person ("I", "my", "I've") and stay in character throughout.
Be warm, genuine, and professionally engaging — as if {NAME} is personally chatting.
Draw on the context provided to give specific, accurate answers.
If something is genuinely not in your context, say so honestly rather than guessing."""

print("✅ Base system prompt set")
print(BASE_SYSTEM_PROMPT)

In [ ]:
# Cell 17: RAG-powered chat function
def build_rag_system_prompt(query: str, k: int = 5) -> str:
    """Retrieve top-k chunks for the query and inject into the system prompt."""
    chunks = ingester.similarity_search(query, k=k)
    if not chunks:
        return BASE_SYSTEM_PROMPT

    context_blocks = []
    for chunk in chunks:
        source = chunk.metadata.get("source", "unknown")
        context_blocks.append(f"[Source: {Path(source).name}]\n{chunk.page_content}")

    context = "\n\n---\n\n".join(context_blocks)
    return BASE_SYSTEM_PROMPT + f"\n\n## Relevant Context\n\n{context}"


def chat(message: str, history: list) -> str:
    """RAG chat: retrieve relevant context per message, then generate a response."""
    # Build context-aware system prompt for this specific query
    system_prompt = build_rag_system_prompt(message)

    # Sanitise history for API compatibility (strip extra Gradio fields)
    clean_history = [
        {"role": h["role"], "content": h["content"]} for h in history
    ]

    messages = (
        [{"role": "system", "content": system_prompt}]
        + clean_history
        + [{"role": "user", "content": message}]
    )

    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
    )
    return response.choices[0].message.content


# Quick sanity check
test_reply = chat("What's your professional background?", [])
print("✅ Chat function working. Sample response:")
print("-" * 60)
print(test_reply[:500])

---
## 🎨 Phase 4 — Gradio UI & HuggingFace Spaces Deployment

In [ ]:
# Cell 19: Opening greeting injected at chat start
GREETING = f"""👋 Hi! I'm a digital twin of **{NAME}** — software engineer, AI builder, and technical writer.

I can answer questions about my background, projects, professional experience, skills, and the things I write and speak about.

Feel free to ask me anything!"""

print("✅ Greeting set:")
print(GREETING)

In [ ]:
# Cell 20: Launch Gradio Chat UI (local preview)
demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    title=f"🤖 {NAME} — Digital Twin",
    description=GREETING,
    examples=[
        "Tell me about your background and experience",
        "What projects are you currently working on?",
        "What's your take on AI and its impact on software?",
        "What technologies do you specialize in?",
        "How can I get in touch with you?",
    ],
    theme=gr.themes.Soft(),
)

demo.launch()

In [ ]:
# Cell 21: Export standalone app.py for HuggingFace Spaces
APP_PY = '''
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

from modules.ingester import (
    Ingester,
    IngestOptions,
    ChunkingOptions,
    VectorStoreOptions,
    EmbeddingOptions,
)

load_dotenv(override=True)
client = OpenAI()

# ─── Config ──────────────────────────────────────────────────────────
NAME          = "David Inyang-Etoh"
KNOWLEDGE_DIR = Path("me/knowledge")
VECTOR_DB_DIR = Path("db/digital_twin_db")
CHAT_MODEL    = "gpt-4o-mini"
EMBED_MODEL   = "text-embedding-3-small"

# ─── Load pre-built vector store (no re-ingestion needed on HF Spaces) ──
ingest_options = IngestOptions(
    knowledge_base_path=str(KNOWLEDGE_DIR),
    chunking=ChunkingOptions(chunk_size=1000, chunk_overlap=150, min_chunk_chars=60),
    vector_store=VectorStoreOptions(
        persist_directory=str(VECTOR_DB_DIR),
        delete_existing_collection=False,  # reuse pre-built DB
    ),
    embedding=EmbeddingOptions(model=EMBED_MODEL),
)
ingester = Ingester(ingest_options)

# Warm up the vector store connection
_, _ = ingester.ingest()

# ─── System prompt ───────────────────────────────────────────────────
BASE_SYSTEM_PROMPT = f"""You are acting as {NAME} — their digital twin.
You answer questions about {NAME}\'s career, skills, projects, writing, and professional interests.
Always speak in first person and stay in character. Be warm and professionally engaging.
If something is not in your context, say so honestly."""


def build_rag_system_prompt(query: str, k: int = 5) -> str:
    chunks = ingester.similarity_search(query, k=k)
    if not chunks:
        return BASE_SYSTEM_PROMPT
    context_blocks = [
        f"[Source: {Path(c.metadata.get(\'source\', \'\'))  .name}]\\n{c.page_content}"
        for c in chunks
    ]
    context = "\\n\\n---\\n\\n".join(context_blocks)
    return BASE_SYSTEM_PROMPT + f"\\n\\n## Relevant Context\\n\\n{context}"


def chat(message: str, history: list) -> str:
    system_prompt = build_rag_system_prompt(message)
    clean_history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_prompt}]
        + clean_history
        + [{"role": "user", "content": message}]
    )
    resp = client.chat.completions.create(model=CHAT_MODEL, messages=messages)
    return resp.choices[0].message.content


# ─── Gradio UI ───────────────────────────────────────────────────────
GREETING = f"""👋 Hi! I\'m a digital twin of **{NAME}** — software engineer, AI builder, and technical writer.

Ask me anything about my background, projects, experience, or ideas!"""

demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    title=f"🤖 {NAME} — Digital Twin",
    description=GREETING,
    examples=[
        "Tell me about your background and experience",
        "What projects are you currently working on?",
        "What\'s your take on AI and software development?",
        "What technologies do you specialize in?",
        "How can I get in touch with you?",
    ],
    theme=gr.themes.Soft(),
)

if __name__ == "__main__":
    demo.launch()
'''

app_path = Path("app.py")
app_path.write_text(APP_PY.strip(), encoding="utf-8")
print(f"✅ app.py written ({app_path.stat().st_size:,} bytes)")
print(f"   Ready for HuggingFace Spaces deployment")

---
## 🚀 HuggingFace Spaces Deployment Checklist

### Files to push to your HF Space:

| File / Folder | Purpose |
|---------------|---------|
| `app.py` | Main entry point (generated by Cell 21) |
| `modules/crawler.py` | Crawler module |
| `modules/ingester.py` | Ingester module |
| `modules/cleaner.py` | Text cleaning utilities |
| `modules/embeddings.py` | Embedding helpers |
| `me/knowledge/` | All `.md` knowledge base files |
| `db/digital_twin_db/` | Pre-built ChromaDB vector store |
| `requirements.txt` | Python dependencies |

### Steps:

1. **Create a new Space** on [huggingface.co/spaces](https://huggingface.co/spaces) → choose **Gradio** SDK
2. **Add Secret**: `OPENAI_API_KEY` in Space Settings → Secrets
3. **Push files** via HF CLI or web upload:
   ```bash
   huggingface-cli login
   huggingface-cli repo create <your-space-name> --type space --space_sdk gradio
   # Then git push the files listed above
   ```
4. Space auto-builds and goes live ✅

### `requirements.txt` content:
```
openai>=1.0.0
gradio>=5.0.0
python-dotenv>=1.0.0
langchain>=0.3.0
langchain-openai>=0.3.0
langchain-chroma>=0.2.0
langchain-text-splitters>=0.3.0
langchain-community>=0.3.0
chromadb>=0.6.0
beautifulsoup4>=4.12.0
lxml>=5.0.0
pyyaml>=6.0.0
pydantic>=2.0.0
pypdf>=5.0.0
```

> **Tip**: The `db/digital_twin_db/` folder contains the pre-built ChromaDB so HF Spaces doesn't need to re-embed on startup — just set `delete_existing_collection=False` in `app.py` (already done).